# Step 1. Check GPU availability

 AlphaFold2 and GNINA can both benefit from GPU acceleration. Before running computational biology tools, we check whether a GPU is available in Google Colab. A GPU reduces the time needed for structure prediction and docking calculations.


In [ ]:

!nvidia-smi

Sun May 31 11:26:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Step 2. Install required libraries and tools

Bioinformatics workflows require multiple tools. Biopython is used to handle DNA, RNA, and protein FASTA files. Open Babel is used to convert molecular file formats. Py3Dmol is used for 3D visualization. Pandas helps organize docking scores into tables.


In [ ]:
# Install required tools
!apt-get update -qq
!apt-get install -y -qq openbabel wget zip
!pip install biopython pandas py3Dmol requests

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# Step 3. Upload FASTA file

The workflow starts with DNA because DNA contains the genetic information. In the central dogma, DNA is transcribed into RNA, and RNA is translated into protein. The input file should contain the EGFR DNA sequence in FASTA format.


In [ ]:
# Upload the file to Google Colab
from google.colab import files
uploaded = files.upload()

Saving egfr_mrna.fasta to egfr_mrna.fasta


In [ ]:
!ls

egfr_mrna.fasta  EGFR_Project  gnina  sample_data


In [ ]:
!pip install biopython

# Step 4. Read the DNA sequence

FASTA files store biological sequences with a header line and sequence lines. Here, the EGFR DNA sequence is read into Python so it can be processed. DNA is made of the bases A, T, G, and C.


In [ ]:
# Read uploaded mRNA FASTA in Colab
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

record = SeqIO.read("egfr_mrna.fasta", "fasta")
egfr_mrna = record.seq

print(record.description)
print("mRNA length:", len(egfr_mrna))
print(egfr_mrna[:100])

NM_005228.5 Homo sapiens epidermal growth factor receptor (EGFR), transcript variant 1, mRNA
mRNA length: 9905
AGACGTCCGGGCAGCCCCCGGCGCAGCGCGGCCGCAGCAGCCTCCGCCCCCCGCACGGTGTGAGCGCCCGACGCGGCCGAGGCGGCCGGAGTCCCGAGCT


# Step 5. Transcribe DNA into mRNA

Transcription converts DNA into messenger RNA. During transcription, thymine (T) in DNA is replaced by uracil (U) in RNA. The mRNA sequence carries the genetic message that will later be translated into amino acids.


In [ ]:
# Convert mRNA to DNA
egfr_dna = Seq(str(egfr_mrna).replace("U", "T"))

dna_record = SeqRecord(
    egfr_dna,
    id="EGFR_DNA",
    description="DNA sequence converted from EGFR mRNA"
)

SeqIO.write(dna_record, "egfr_dna.fasta", "fasta")

print("DNA length:", len(egfr_dna))
print(egfr_dna[:100])

DNA length: 9905
AGACGTCCGGGCAGCCCCCGGCGCAGCGCGGCCGCAGCAGCCTCCGCCCCCCGCACGGTGTGAGCGCCCGACGCGGCCGAGGCGGCCGGAGTCCCGAGCT


In [ ]:
# Convert DNA back to mRNA
egfr_mrna_converted = egfr_dna.transcribe()

mrna_record = SeqRecord(
    egfr_mrna_converted,
    id="EGFR_mRNA_converted",
    description="mRNA transcribed from EGFR DNA"
)

SeqIO.write(mrna_record, "egfr_mrna_converted.fasta", "fasta")

print("Converted mRNA length:", len(egfr_mrna_converted))
print(egfr_mrna_converted[:100])

Converted mRNA length: 9905
AGACGUCCGGGCAGCCCCCGGCGCAGCGCGGCCGCAGCAGCCUCCGCCCCCCGCACGGUGUGAGCGCCCGACGCGGCCGAGGCGGCCGGAGUCCCGAGCU


# Step 6. Translate mRNA into protein

Translation converts the mRNA codon sequence into an amino acid sequence. Each codon contains three RNA bases and usually codes for one amino acid. The resulting protein sequence is required for AlphaFold2 structure prediction.


In [ ]:
# Upload protein FASTA to Colab
uploaded = files.upload()

Saving egfr_protein.fasta to egfr_protein.fasta


In [ ]:
# read protein
protein_record = SeqIO.read("egfr_protein.fasta", "fasta")
egfr_protein = protein_record.seq

print(protein_record.description)
print("Protein length:", len(egfr_protein))
print(egfr_protein[:100])

XP_054213393.1 epidermal growth factor receptor isoform X1 [Homo sapiens]
Protein length: 1157
MFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNN


In [ ]:
# Use this protein for BLAST
print(">EGFR_protein")
print(str(egfr_protein))

>EGFR_protein
MFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCNLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLEGCPTNGPKIPSIATGMVGALLLLLVVALGIGLFMRRRHIVRKRTLRRLLQERELVEPLTPSGEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQGFFSSPSTSRTPLLSSLS

# Step 7. Create the project folder structure

A clear folder structure keeps input sequences, predicted structures, scripts, docking files, and result images separate. This makes the notebook easier to understand, repeat, and submit as a project.


In [ ]:
# Create project folders
import os

folders = [
    "EGFR_Project",
    "EGFR_Project/data",
    "EGFR_Project/structures",
    "EGFR_Project/scripts",
    "EGFR_Project/docking_results",
    "EGFR_Project/alphafold_results",
    "EGFR_Project/images"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

!find EGFR_Project -maxdepth 2 -type d

EGFR_Project
EGFR_Project/data
EGFR_Project/scripts
EGFR_Project/docking_results
EGFR_Project/alphafold_results
EGFR_Project/images
EGFR_Project/structures


# Step 8. Save sequence files into the project folder

The DNA, mRNA, and protein FASTA files are copied into the project data folder. These files represent the biological flow from gene sequence to final protein sequence.


In [ ]:
# Move your files into the project folder:
!cp egfr_mrna.fasta EGFR_Project/data/
!cp egfr_dna.fasta EGFR_Project/data/
!cp egfr_mrna_converted.fasta EGFR_Project/data/
!cp egfr_protein.fasta EGFR_Project/data/

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.4 MB/s eta 0:00:00


In [ ]:
!find . -name "*protein*.fasta" -o -name "*.fasta"

In [ ]:
from google.colab import files
uploaded = files.upload()
!ls -lh

Saving egfr_protein.fasta to egfr_protein (1).fasta
total 16K
drwxr-xr-x 3 root root 4.0K May 31 16:59  EGFR_Project
-rw-r--r-- 1 root root 1.3K May 31 17:03 'egfr_protein (1).fasta'
-rw-r--r-- 1 root root 1.3K May 31 17:02  egfr_protein.fasta
drwxr-xr-x 1 root root 4.0K May 26 13:31  sample_data


In [ ]:
!cp "egfr_protein (1).fasta" EGFR_Project/data/egfr_protein.fasta

In [ ]:
!ls -lh EGFR_Project/data/

total 4.0K
-rw-r--r-- 1 root root 1.3K May 31 17:07 egfr_protein.fasta


# Step 9. Prepare the EGFR kinase-domain FASTA for AlphaFold2

AlphaFold2 predicts a 3D protein structure from an amino acid sequence. EGFR is a large receptor protein, so the kinase domain is commonly selected because it contains the drug-binding site targeted by inhibitors such as erlotinib. The EGFR kinase domain is approximately residues 300–1200.


In [ ]:
# create the AlphaFold2 kinase-domain FASTA:
from Bio import SeqIO

protein_record = SeqIO.read("EGFR_Project/data/egfr_protein.fasta", "fasta")
egfr_protein = protein_record.seq

egfr_full_protein = str(egfr_protein)

# EGFR kinase domain residues 671–998
egfr_kinase_domain = egfr_full_protein[670:998]

with open("EGFR_Project/data/egfr_kinase_domain_for_alphafold.fasta", "w") as f:
    f.write(">EGFR_kinase_domain_residues_671_998\n")
    f.write(egfr_kinase_domain + "\n")

print("Full protein length:", len(egfr_full_protein))
print("Kinase domain length:", len(egfr_kinase_domain))
print(egfr_kinase_domain[:100])

Full protein length: 1157
Kinase domain length: 328
GTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAK


# Step 10. Display the AlphaFold2 input sequence

Before submitting a sequence to AlphaFold2 or ColabFold, it is good practice to print and verify the FASTA sequence. This ensures that the correct protein region is being used for structure prediction.


In [ ]:
print(">EGFR_kinase_domain_residues_671_998")
print(egfr_kinase_domain)

>EGFR_kinase_domain_residues_671_998
GTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQGFFSSPSTSRTPLLSSLSATSNNSTVACID


# Step 11. Upload the AlphaFold2 predicted PDB file

AlphaFold2 produces predicted protein structures in PDB format. After running AlphaFold2 or ColabFold using the kinase-domain FASTA sequence, upload the predicted PDB file here so it can be stored and visualized in this notebook.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving EGFR_kinase_domain_AlphaFold2_c2d8a_0_unrelaxed_rank_002_alphafold2_ptm_model_1_seed_000.pdb to EGFR_kinase_domain_AlphaFold2_c2d8a_0_unrelaxed_rank_002_alphafold2_ptm_model_1_seed_000.pdb


# Step 12. Move AlphaFold2 structures into the results folder

The predicted PDB file is moved into the AlphaFold2 results folder. This separates predicted structures from downloaded experimental structures and keeps the project organized.


In [ ]:
import os, glob, shutil

os.makedirs("EGFR_Project/alphafold_results", exist_ok=True)

for pdb in glob.glob("*.pdb"):
    shutil.move(pdb, "EGFR_Project/alphafold_results/" + pdb)
    print("Moved:", pdb)


Moved: EGFR_kinase_domain_AlphaFold2_c2d8a_0_unrelaxed_rank_002_alphafold2_ptm_model_1_seed_000.pdb


In [ ]:
!ls -lh EGFR_Project/alphafold_results/

total 208K
-rw-r--r-- 1 root root 207K May 31 18:07 EGFR_kinase_domain_AlphaFold2_c2d8a_0_unrelaxed_rank_002_alphafold2_ptm_model_1_seed_000.pdb


# Step 13. Visualize the AlphaFold2 predicted structure

Protein visualization helps confirm the overall folding pattern of the predicted EGFR kinase domain. Py3Dmol displays the 3D structure as a cartoon model, making helices, sheets, and loops easier to inspect.


In [ ]:
!pip install py3Dmol

In [ ]:
# Visualize AlphaFold2 structure
import py3Dmol
import glob

af_files = glob.glob("EGFR_Project/alphafold_results/*.pdb")

af_pdb = af_files[0]
print("Showing:", af_pdb)

with open(af_pdb) as f:
    af_structure = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(af_structure, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

Showing: EGFR_Project/alphafold_results/EGFR_kinase_domain_AlphaFold2_c2d8a_0_unrelaxed_rank_002_alphafold2_ptm_model_1_seed_000.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Step 14. Download an EGFR crystal structure for GNINA docking

GNINA requires a receptor structure and a ligand structure. The PDB structure 1M17 contains EGFR kinase domain bound to erlotinib, so it is useful for preparing the receptor, ligand, and docking box. This docking section is placed after AlphaFold2 as requested.


In [ ]:
# Download EGFR structure for docking
!wget -q https://files.rcsb.org/download/1M17.pdb -O EGFR_Project/structures/egfr_1m17.pdb

!head EGFR_Project/structures/egfr_1m17.pdb

HEADER    TRANSFERASE                             17-JUN-02   1M17              
TITLE     EPIDERMAL GROWTH FACTOR RECEPTOR TYROSINE KINASE DOMAIN WITH 4-       
TITLE    2 ANILINOQUINAZOLINE INHIBITOR ERLOTINIB                               
COMPND    MOL_ID: 1;                                                            
COMPND   2 MOLECULE: EPIDERMAL GROWTH FACTOR RECEPTOR;                          
COMPND   3 CHAIN: A;                                                            
COMPND   4 FRAGMENT: TYROSINE KINASE DOMAIN (RESIDUES 671-998);                 
COMPND   5 SYNONYM: RECEPTOR PROTEIN-TYROSINE KINASE ERBB-1;                    
COMPND   6 EC: 2.7.1.112;                                                       
COMPND   7 ENGINEERED: YES                                                      


In [ ]:
!ls -lh EGFR_Project/structures/

total 248K
-rw-r--r-- 1 root root 247K May 31 12:46 egfr_1m17.pdb


# Step 15. Separate receptor and ligand from the PDB file

Molecular docking needs the protein receptor and small-molecule ligand as separate files. In this structure, erlotinib is recorded as the ligand AQ4. Protein atoms are saved as the receptor, while AQ4 atoms are saved as the ligand.


In [ ]:
# Prepare receptor and ligand
# The ligand erlotinib in 1M17 is usually named AQ4.
pdb_file = "EGFR_Project/structures/egfr_1m17.pdb"

receptor_lines = []
ligand_lines = []

with open(pdb_file) as f:
    for line in f:
        if line.startswith("ATOM"):
            receptor_lines.append(line)
        elif line.startswith("HETATM") and "AQ4" in line:
            ligand_lines.append(line)

with open("EGFR_Project/structures/egfr_receptor.pdb", "w") as f:
    f.writelines(receptor_lines)
    f.write("END\n")

with open("EGFR_Project/structures/erlotinib_from_pdb.pdb", "w") as f:
    f.writelines(ligand_lines)
    f.write("END\n")

print("Receptor atoms:", len(receptor_lines))
print("Ligand atoms:", len(ligand_lines))

Receptor atoms: 2511
Ligand atoms: 29


# Step 16. Install and check GNINA

GNINA is a deep-learning-based molecular docking program. It predicts possible ligand binding poses and scores them using docking affinity and convolutional neural network scoring methods.


In [ ]:
# Install Open Babel and GNINA
!apt-get update -qq
!apt-get install -y -qq openbabel wget zip
!pip install pandas py3Dmol biopython

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!wget -q https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2 -O gnina
!chmod +x gnina
!./gnina --help | head


Input:
  -r [ --receptor ] arg              rigid part of the receptor
  --flex arg                         flexible side chains, if any (PDBQT)
  -l [ --ligand ] arg                ligand(s)
  --flexres arg                      flexible side chains specified by comma 
                                     separated list of chain:resid
  --flexdist_ligand arg              Ligand to use for flexdist
  --flexdist arg                     set all side chains within specified 
                                     distance to flexdist_ligand to flexible


# Step 17. Convert the ligand into SDF format

Ligands can be stored in different chemical file formats. GNINA can work with ligand files such as SDF. Open Babel converts the extracted erlotinib PDB ligand into SDF and generates 3D chemical information if needed.


In [ ]:
# Convert ligand to SDF
!obabel EGFR_Project/structures/erlotinib_from_pdb.pdb \
  -O EGFR_Project/structures/erlotinib.sdf \
  --gen3d

!ls -lh EGFR_Project/structures/erlotinib.sdf

1 molecule converted
-rw-r--r-- 1 root root 4.9K May 31 13:00 EGFR_Project/structures/erlotinib.sdf


# Step 18. Calculate the docking box center

Docking is performed inside a defined search box. Because erlotinib is already bound in the crystal structure, its coordinates can be used to estimate the center of the active site. GNINA will search for ligand poses around this region.


In [ ]:
# Find docking box center
import numpy as np

coords = []

with open("EGFR_Project/structures/erlotinib_from_pdb.pdb") as f:
    for line in f:
        if line.startswith("HETATM"):
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            coords.append([x, y, z])

coords = np.array(coords)
center = coords.mean(axis=0)

center_x, center_y, center_z = center

print("center_x =", round(center_x, 3))
print("center_y =", round(center_y, 3))
print("center_z =", round(center_z, 3))

center_x = 22.014
center_y = 0.253
center_z = 52.794


# Step 19. Run GNINA docking

GNINA places the ligand into the receptor binding pocket and evaluates possible binding poses. The output SDF file contains predicted poses, while the log file contains docking scores such as affinity, CNN score, and CNN affinity.


In [ ]:
# Run GNINA docking
!./gnina \
  --receptor EGFR_Project/structures/egfr_receptor.pdb \
  --ligand EGFR_Project/structures/erlotinib.sdf \
  --center_x {center_x} \
  --center_y {center_y} \
  --center_z {center_z} \
  --size_x 20 \
  --size_y 20 \
  --size_z 20 \
  --out EGFR_Project/docking_results/egfr_erlotinib_docked.sdf \
  --log EGFR_Project/docking_results/gnina_log.txt

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul  8 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina --receptor EGFR_Project/structures/egfr_receptor.pdb --ligand EGFR_Project/structures/erlotinib.sdf --center_x 22.013689655172417 --center_y 0.2528275862068965 --center_z 52.79403448275863 --size_x 20 --size_y 20 --size_z 20 --out EGFR_Project/docking_results/egfr_erlotinib_docked.sdf --log EGFR_Project/docking_results/gnina_log.txt
Using random seed: 116551492

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
EGFR_Project/structures/erlotinib_from_pdb.pdb | pose 0 | initial pose not within box

mode |  affinit

# Step 20. View GNINA docking scores

Docking scores help compare predicted ligand poses. More negative affinity values usually suggest stronger predicted binding, while CNN score and CNN affinity provide GNINA neural-network-based pose and binding estimates.


In [ ]:
# View docking scores
!cat EGFR_Project/docking_results/gnina_log.txt | tail -60

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul  8 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina --receptor EGFR_Project/structures/egfr_receptor.pdb --ligand EGFR_Project/structures/erlotinib.sdf --center_x 22.013689655172417 --center_y 0.2528275862068965 --center_z 52.79403448275863 --size_x 20 --size_y 20 --size_z 20 --out EGFR_Project/docking_results/egfr_erlotinib_docked.sdf --log EGFR_Project/docking_results/gnina_log.txt
Using random seed: 116551492

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affinity
-----+------------+------------+------------+----------
    1       -7.00       -0.72       0.9207      7.360
    2       -7.22       -0.90       0.9141  

# Step 21. Save docking scores as a CSV file

Saving results in CSV format makes them easier to include in reports, tables, and graphs. Each docking mode is stored with its predicted affinity and GNINA CNN-based scores.


In [ ]:
# Save docking scores as CSV
import re
import pandas as pd

log_file = "EGFR_Project/docking_results/gnina_log.txt"
rows = []

with open(log_file) as f:
    for line in f:
        line = line.strip()
        if re.match(r"^\d+\s+", line):
            parts = line.split()
            if len(parts) >= 4:
                rows.append({
                    "mode": parts[0],
                    "affinity_kcal_mol": parts[1],
                    "cnn_score": parts[2],
                    "cnn_affinity": parts[3]
                })

df = pd.DataFrame(rows)
df.to_csv("EGFR_Project/docking_results/docking_scores.csv", index=False)
df

,mode,affinity_kcal_mol,cnn_score,cnn_affinity
0,1,-7.00,-0.72,0.9207
1,2,-7.22,-0.90,0.9141
2,3,-7.48,-0.03,0.8995
3,4,-6.40,-0.87,0.8343
4,5,-6.34,-1.13,0.8194
5,6,-7.28,-0.55,0.7370
6,7,-7.16,-0.69,0.7227
7,8,-6.66,-0.62,0.6619
8,9,-6.49,-1.15,0.6589


# Step 22. Visualize the docked EGFR–erlotinib complex

Visualizing the receptor and docked ligand helps interpret whether the ligand is positioned inside the expected binding pocket. The receptor is shown as a cartoon and the ligand is shown as sticks.


In [ ]:
# Visualize docking result
import py3Dmol

with open("EGFR_Project/structures/egfr_receptor.pdb") as f:
    receptor = f.read()

with open("EGFR_Project/docking_results/egfr_erlotinib_docked.sdf") as f:
    docked_ligand = f.read()

view = py3Dmol.view(width=800, height=600)

view.addModel(receptor, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightblue"}})

view.addModel(docked_ligand, "sdf")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.